# ARC-AGI-3 Synesthesia R5 native untimed

**Arm** of `codywhatleymd/arc-agi-3-synesthesia`: `ARM_B_STUDENT_TOWER=0`,
`ARM_EXTRA_VLLM_ARGS=--mm-processor-cache-gb 0`, `ARM_LIMIT_MM={"image":143,"video":0}` (harness switches,
harness/SWITCHES.md); harness dataset `codywhatleymd/arc3-synesthesia-harness-r0`. Everything else is that notebook.

Synesthesia sends the board to the model through its visual pathway. Our board-tower student, an 18.2M-parameter
network distilled from the model's own 449M-parameter vision tower, encodes each 64x64 board straight into 1,024
image-embedding rows, which vLLM accepts as `image_embeds`. No image is rendered and the native vision tower is never
loaded: each cell's color reaches the model as a visual percept, the way a grapheme-color synesthete sees letters.

What runs here, all of it ours:

- **Model:** our NVFP4/FP8 quantization of Qwen3.8-Flash-Next with an NVFP4 multi-token-prediction head
  (Kaggle Model `codywhatleymd/arc3-flashnext-nvfp4-ours-v22`, v22/3).
- **Serving:** our vLLM 0.31.0 runtime, rebuilt for Python 3.13 (`codywhatleymd/vllm-v0310-arc3-py313`), with bounded
  KV block copies, shipped autotune tactics and parallel shard prefetch for a fast, stable cold start.
- **Board channel:** the board-tower student (`codywhatleymd/arc3-board-tower-armb`) and the harness changes that carry
  its embeddings and price them correctly in the context budget.
- **Harness:** our own tree (`codywhatleymd/arc3-synesthesia-harness-r0`, built from one commit of our repository). It is
  copied and run as it is; nothing is patched at run time.

Built on the TAAF ARC-AGI-3 agent harness (Tufa ARC-AGI Framework), serving on vLLM.


In [ ]:
# ================================ BOOT: printed before any input is touched =======================
import faulthandler, json, os, pathlib, platform, sys, time
_BOOT_T0 = time.time()
print("ARM: ARC-AGI-3 Synesthesia R5 native untimed env: ARM_B_STUDENT_TOWER=0 ARM_EXTRA_VLLM_ARGS=--mm-processor-cache-gb 0 ARM_LIMIT_MM={\"image\":143,\"video\":0}", flush=True)
print(f"BOOT {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())} UTC | python {platform.python_version()} | "
      f"{platform.platform()} | pid {os.getpid()}", flush=True)
_BOOT_SAFE_ENV = ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_IS_COMPETITION_RERUN", "KAGGLE_DOCKER_IMAGE",
                  "KAGGLE_GPU_TYPE", "KAGGLE_CONTAINER_NAME")
print("BOOT env:", {k: os.environ[k] for k in _BOOT_SAFE_ENV if k in os.environ},
      "| other KAGGLE_* names (values not printed):",
      sorted(k for k in os.environ if k.startswith("KAGGLE_") and k not in _BOOT_SAFE_ENV), flush=True)


def _boot_installed(name: str):
    try:
        from importlib import metadata
        return metadata.version(name)
    except Exception:
        return None


# A fingerprint of the container image (read from package metadata, nothing imported), to compare a Kaggle
# session's image with the local test image gcr.io/kaggle-gpu-images/python:latest.
print("BOOT image packages:", {n: _boot_installed(n) for n in
                               ("torch", "numpy", "pillow", "papermill", "ipykernel", "jupyter_client")}, flush=True)
faulthandler.dump_traceback_later(60, repeat=True)
_BOOT_SOURCES = json.loads(r'''[["dataset", "codywhatleymd/arc3-synesthesia-harness-r0"], ["dataset", "codywhatleymd/arc3-board-tower-armb"], ["dataset", "codywhatleymd/vllm-v0310-arc3-py313"], ["model", "codywhatleymd/arc3-flashnext-nvfp4-ours-v22/PyTorch/v22/3"], ["competition", "arc-prize-2026-arc-agi-3"]]''')


def _boot_mount_paths(kind: str, ref: str) -> list:
    root = pathlib.Path("/kaggle/input")
    if kind == "dataset":
        owner, slug = ref.split("/")
        return [root / "datasets" / owner / slug, root / slug]
    if kind == "model":
        owner, model, framework, instance, version = ref.split("/")
        frameworks = list(dict.fromkeys((framework.lower(), framework)))
        return ([root / "models" / owner / model / f / instance / version for f in frameworks]
                + [root / model / f / instance / version for f in frameworks])
    return [root / "competitions" / ref, root / ref]


for _kind, _ref in _BOOT_SOURCES:
    _t = time.time()
    _paths = _boot_mount_paths(_kind, _ref)
    _mount = next((p for p in _paths if p.is_dir()), None)
    if _mount is None:
        print(f"BOOT input {_kind} {_ref}: NOT MOUNTED (tried {[str(p) for p in _paths]})", flush=True)
        continue
    _entries = sorted(os.scandir(_mount), key=lambda e: e.name)
    _first = next((e for e in _entries if e.is_file()), None)
    _read = 0
    if _first is not None:
        with open(_first.path, "rb") as _f:
            _read = len(_f.read(1 << 20))
    print(f"BOOT input {_kind} {_ref}: {_mount} | {len(_entries)} entries | read {_read:,} B of "
          f"{_first.name if _first else '-'} | {time.time() - _t:.2f}s", flush=True)
faulthandler.cancel_dump_traceback_later()
print(f"BOOT inputs checked in {time.time() - _BOOT_T0:.1f}s", flush=True)


In [ ]:
# ================================ Synesthesia: harness + environment ================================
# The harness is ours: harness/ in the arc-prize-2026 repository, shipped as a dataset built from one git commit. It is
# copied to a writable directory and run as it is; nothing is patched here.
import hashlib, importlib.util, json, os, pathlib, shutil, sys

ARM_LABEL = "synesthesia"                      # output folder name (/kaggle/working/synesthesia)
EXPECT_HARNESS_COMMIT = "85529646afc42a5a0b5cae004501a917d0167f91"
EXPECT_CKPT_SHA = "5e4eb805a80cab0f21b67dde0240e55d1216d9999fbe1dcd4043a2b544db445b"
MODEL_PARTS = ("codywhatleymd", "arc3-flashnext-nvfp4-ours-v22", "PyTorch", "v22", "3")


def _dataset_dir(owner: str, slug: str, marker: str) -> pathlib.Path:
    """Kaggle mounts a dataset at /kaggle/input/datasets/<owner>/<slug> or /kaggle/input/<slug>; check both."""
    for cand in (pathlib.Path("/kaggle/input/datasets") / owner / slug, pathlib.Path("/kaggle/input") / slug):
        if (cand / marker).is_file():
            return cand
    raise RuntimeError(f"dataset {owner}/{slug} is not mounted (no {marker} at either mount path)")


# ---- 1. the harness: our commit, copied, nothing patched -------------------------------------------------------
HARNESS_SRC = _dataset_dir("codywhatleymd", "arc3-synesthesia-harness-r0", "taaf-kaggle-bundle.json")
_commit = (HARNESS_SRC / "HARNESS_COMMIT.txt").read_text().split()[0]
if not _commit.startswith(EXPECT_HARNESS_COMMIT):
    raise RuntimeError(f"harness dataset is commit {_commit}, this notebook was built for {EXPECT_HARNESS_COMMIT}")
HARNESS_DIR = pathlib.Path("/tmp/synesthesia_harness")   # writable, and not a notebook output
if HARNESS_DIR.exists():
    shutil.rmtree(HARNESS_DIR)
shutil.copytree(HARNESS_SRC, HARNESS_DIR)
_n = 0
for _src in HARNESS_SRC.rglob("*"):
    if _src.is_file():
        _n += 1
        if (HARNESS_DIR / _src.relative_to(HARNESS_SRC)).read_bytes() != _src.read_bytes():
            raise RuntimeError(f"harness copy differs from the dataset at {_src.relative_to(HARNESS_SRC)}")
print(f"synesthesia: harness {HARNESS_SRC} @ {_commit[:12]} -> {HARNESS_DIR} ({_n} files, copied as is)", flush=True)

# ---- 2. the board tower: the checkpoint is byte-identical to the one v8 ran -------------------------------------
BT = _dataset_dir("codywhatleymd", "arc3-board-tower-armb", "SHA256SUMS")
CKPT = BT / "ckpt_px16_dih" / "best.pt"
h = hashlib.sha256()
with open(CKPT, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
if h.hexdigest() != EXPECT_CKPT_SHA:
    raise RuntimeError(f"board-tower checkpoint sha256 {h.hexdigest()} != {EXPECT_CKPT_SHA}")
print(f"synesthesia: tower {BT}, ckpt sha256 OK ({EXPECT_CKPT_SHA[:16]}..., {CKPT.stat().st_size:,} B)", flush=True)


# ---- 3. the model: the Kaggle Model, mounted in place, found the way the setup script finds it ------------------
def _model_dir(owner: str, model: str, framework: str, instance: str, version: str) -> pathlib.Path:
    cands = []
    for fw in dict.fromkeys((framework.lower(), framework)):
        cands += [pathlib.Path("/kaggle/input/models") / owner / model / fw / instance / version,
                  pathlib.Path("/kaggle/input") / model / fw / instance / version]
    for cand in cands:
        if (cand / "model.safetensors.index.json").is_file():
            return cand
    raise RuntimeError(f"model {'/'.join((owner, model, framework, instance, version))} is not mounted "
                       f"(tried {[str(c) for c in cands]})")


MODEL_DIR = _model_dir(*MODEL_PARTS)
_wm = json.loads((MODEL_DIR / "model.safetensors.index.json").read_text())["weight_map"]
_need = sorted(set(_wm.values()))
_miss = [f for f in _need if not (MODEL_DIR / f).is_file()]
if _miss:
    raise RuntimeError(f"{MODEL_DIR}: {len(_miss)} shard files missing, e.g. {_miss[:3]}")
print(f"synesthesia: model {MODEL_DIR} ({len(_need)} shard files, "
      f"{sum((MODEL_DIR / f).stat().st_size for f in _need) / 1e9:.1f} GB, {len(_wm)} tensors)", flush=True)

# ---- 4. the embedding channel is in this harness ----------------------------------------------------------------
MARKERS = {
    "setup_commands.json": ["ARM_MAX_PIXELS", "ARM_LIMIT_MM", "ARM_ENABLE_MM_EMBEDS", "ARM_UPSCALE",
                            "vllm-v0310-arc3-py313", "'arc3_tools' / 'patch_kv_copy_chunked.py'", "_ARC3_PF_NEW"],
    "src/ARC3-Inference/inference/agent/token_estimate.py": ["_embeds_part_tokens", '"image_embeds"'],
    "src/ARC3-Inference/inference/agent/transcript_log.py": ["_elide_embeds_payloads"],
    "src/ARC3-Inference/inference/agent/vision_context.py":
        ["_student_tower_part", "ARM_B_STUDENT_TOWER", "BOARD_TOWER_THREADS", "_log_png_part", "ARM_B_EXPECT_NPOS"],
}
for rel, needles in MARKERS.items():
    text = (HARNESS_DIR / rel).read_text()
    for needle in needles:
        if needle not in text:
            raise RuntimeError(f"{rel}: {needle!r} absent -- this is not the Synesthesia harness")
if not (HARNESS_DIR / "arc3_tools" / "patch_kv_copy_chunked.py").is_file():
    raise RuntimeError("arc3_tools/patch_kv_copy_chunked.py is missing from the harness")
print("synesthesia: embedding channel present (setup flags, token pricing, log elision, board-tower branch)", flush=True)

# ---- 5. the token-estimate fix, measured before a single game is played -----------------------------------------
# An image_embeds part carries a ~7 MB base64 tensor. Priced as text it would be ~2.4M tokens against a true 1,024,
# and the 81,536-token context target would trim history to nothing every turn. The harness prices it at its row
# count; measured here, not assumed. (torch is not imported yet: the setup may replace it.)
_spec = importlib.util.spec_from_file_location(
    "te_synesthesia", HARNESS_DIR / "src/ARC3-Inference/inference/agent/token_estimate.py")
_te_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_te_mod)
_B64_LEN = 6992612          # measured: torch.save of a (1024, 2560) bfloat16 tensor, base64'd
_fake = {"type": "image_embeds", "uuid": "tower-g1024-fa0a0f5dbd-" + "0" * 40,
         "image_embeds": {"image_embeds": "A" * _B64_LEN, "image_grid_thw": "A" * 300}}
_msgs = [{"role": "user", "content": [{"type": "text", "text": "Pick the next action."}, _fake]}]
_priced_total = _te_mod.estimate_tokens(_msgs)
_rows = _te_mod._embeds_part_tokens(_fake)
print(f"synesthesia: token estimate of one board message = {_priced_total:,} tokens (board priced at {_rows}, "
      f"true 1024)", flush=True)
if _rows != 1024 or _priced_total > 2000:
    raise RuntimeError(f"token estimate prices a board at {_rows} rows / {_priced_total} tokens -- history would be "
                       f"trimmed to nothing")

# ---- 6. the run's environment -----------------------------------------------------------------------------------
# Every variable gates one harness behaviour and defaults to the base harness's value when unset.
ARM_ENV = {
    "ARM_LABEL": ARM_LABEL,
    "ARM_UPSCALE": "16",                 # 1024 tokens per board (16 px per cell)
    "ARM_MAX_PIXELS": "1048576",         # -> --mm-processor-kwargs {"max_pixels": 1048576}
    "ARM_LIMIT_MM": '{"image":0,"video":0}',   # no image may arrive as pixels: the native tower is never loaded
    "ARM_ENABLE_MM_EMBEDS": "1",         # -> --enable-mm-embeds
    "ARM_B_STUDENT_TOWER": "1",          # the board-tower branch of vision_context
    "ARM_B_EXPECT_NPOS": "1024",         # hard-fails a geometry-mismatched checkpoint
    "BOARD_TOWER_REPO": str(BT),
    "BOARD_TOWER_CKPT": str(CKPT),
    "BOARD_TOWER_DEVICE": "cpu",         # the GPU belongs to the server
    "BOARD_TOWER_ANY_GPU": "1",
    "BOARD_TOWER_THREADS": "2",          # an uncapped torch takes every core and starves the server
    "TAAF_KAGGLE_BUNDLE_DIR": str(HARNESS_DIR),
    # A commit run's whole budget from notebook start (the pickled default, 2100 s, leaves almost no play after the
    # setup and the model load). A scored rerun ignores it: its soft end is None.
    "ARM_MAX_RUNTIME_S": "0",
}
# ---- the arm: its own kernel, harness dataset and switches (build_synesthesia_notebook.py arm flags) -------------
# The switches are set on top of the environment above; this arm REDEFINES two of its values on purpose
# (ARM_B_STUDENT_TOWER, ARM_LIMIT_MM): the native tower replaces the student.
ARM_NAME = "ARC-AGI-3 Synesthesia R5 native untimed"
ARM_KERNEL = "codywhatleymd/arc-agi-3-synesthesia-r5-native-untimed"
ARM_HARNESS_DATASET = "codywhatleymd/arc3-synesthesia-harness-r0"
ARM_SWITCHES = {"ARM_B_STUDENT_TOWER": "0", "ARM_EXTRA_VLLM_ARGS": "--mm-processor-cache-gb 0", "ARM_LIMIT_MM": "{\"image\":143,\"video\":0}"}
ARM_ENV.update(ARM_SWITCHES)
os.environ.update(ARM_ENV)
for k in sorted(ARM_ENV):
    print(f"synesthesia: env {k}={ARM_ENV[k]}", flush=True)
print(f"synesthesia: host has {os.cpu_count()} vCPU "
      f"(board-tower peak demand {14 * int(ARM_ENV['BOARD_TOWER_THREADS'])} threads)", flush=True)


In [ ]:
import contextlib
import dataclasses
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# The competition mounts at /kaggle/input/competitions/<slug> on some hosts and at /kaggle/input/<slug> on others
# (10-31: the pool-clock run got the second, skipped the arc-agi install and died on a missing arcengine 1,597 s in).
_WHEELHOUSE_CANDIDATES = [Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"),
                          Path("/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")]
wheelhouse = next((p for p in _WHEELHOUSE_CANDIDATES if p.exists()), _WHEELHOUSE_CANDIDATES[0])
print(f"synesthesia: competition wheelhouse {wheelhouse} (exists={wheelhouse.exists()})", flush=True)
if not wheelhouse.exists() and os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    raise RuntimeError(f"competition wheelhouse not found at {[str(p) for p in _WHEELHOUSE_CANDIDATES]}: "
                       f"arc-agi would not be installed and arcengine would be missing")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
DATASET_SOURCES: list[str] = ["codywhatleymd/arc3-synesthesia-harness-r0", "codywhatleymd/vllm-v0310-arc3-py313"]
KERNEL_SOURCES: list[str] = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Audit attached datasets
import subprocess

subprocess.run(["ls", "/kaggle/input"], check=False)

In [ ]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return
    commands = json.loads(path.read_text(encoding="utf-8"))
    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(str(command), shell=True, check=check, cwd=WORKING_DIR, env=env)
        if not check and result.returncode != 0:
            print(f"taaf.kaggle: {label} command exited with {result.returncode}", flush=True)
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Run deployment setup commands before the benchmark pickle is loaded.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# ================================ customization hook ================================
# The harness's customization point: runs after its setup (so torch is whatever the server's install
# left behind) and before the first game.
import time as _time

# ---- this run's artifacts in one directory ----------------------------------------------------
ARM_JOB = WORKING_DIR / ARM_LABEL
ARM_JOB.mkdir(parents=True, exist_ok=True)
bm.job_dir = ARM_JOB

# ---- request logs are the liveness proof, but only affordable off a submission ------------------
bm.solver.save_request_logs = not true_submission
print(f"synesthesia: job_dir={bm.job_dir} save_request_logs={bm.solver.save_request_logs}", flush=True)

# ---- restoration series: solver settings (inert in the anchor; builder --per-game-none etc.) ----
bm.solver.max_runtime_s_per_game = None
bm.solver.concurrency = 14
bm.solver.max_actions_per_game = 5184
print(f"synesthesia: restoration settings: per_game_s={bm.solver.max_runtime_s_per_game} concurrency={bm.solver.concurrency} max_actions={bm.solver.max_actions_per_game} audit_every_actions=None", flush=True)

# ---- OPTIONAL, and inert unless set: lengthen a NON-submission smoke run -----------------------
# The pickled target.max_runtime_s is 2100 s for the WHOLE job and `_soft_end_time` subtracts a
# 600 s buffer, so a commit run has 1500 s from NOTEBOOK START -- setup and the model load come out
# of that, which can leave little or no play time. A submission is untouched either way:
# `_soft_end_time` returns None when run_as_submission, and the guard below is explicit about it.
_mr = os.environ.get("ARM_MAX_RUNTIME_S", "").strip()
if _mr and not true_submission:
    _was = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    target.max_runtime_s = float(_mr)
    soft_end = _soft_end_time(target.max_runtime_s, run_as_submission=run_as_submission)
    print(f"synesthesia: target.max_runtime_s {_was} -> {target.max_runtime_s} "
          f"(non-submission only); soft_end={soft_end}", flush=True)
else:
    print(f"synesthesia: target.max_runtime_s left at the pickled "
          f"{float(getattr(target, 'max_runtime_s', 0.0) or 0.0)}; soft_end={soft_end}", flush=True)

# ---- the board-tower student, loaded once here so a bad mount fails BEFORE any game -------------
# The harness's own _bt_init() imports this same module later and reuses this cached instance.
import torch as _torch
_torch.set_num_threads(int(os.environ["BOARD_TOWER_THREADS"]))
try:
    _torch.set_num_interop_threads(1)
except Exception:
    pass
sys.path.insert(0, os.environ["BOARD_TOWER_REPO"])
from scripts.board_tower import export_embeds as _bt
from notebooks_duck.synth_tokens import synth_embeds as _se

_t0 = _time.time()
_st = _bt.load_student()
print(f"synesthesia: student loaded in {_time.time()-_t0:.1f}s  torch={_torch.__version__} "
      f"ckpt={_st.path} tag={_st.tag} step={_st.step}", flush=True)
print(f"synesthesia: n_pos={_st.n_pos} side={_st.side} device={_st.device} "
      f"grid_thw={_st.grid_thw().tolist()} params={_st.model.count_params()['total']:,} "
      f"cell_patch={_st.model.cfg.cell_patch}", flush=True)
assert _st.n_pos == int(os.environ["ARM_B_EXPECT_NPOS"]), _st.n_pos
assert (_se.PATCH, _se.MERGE, _se.HIDDEN) == (16, 2, 2560)

# one real forward: shape/dtype/finiteness asserts, the wire payload, and its true token count
import numpy as _np
_board = _np.random.default_rng(0).integers(0, 16, (64, 64))
_cells = _se.board_cells(_board)
_t0 = _time.perf_counter()
_emb = _st.tokens(_board)
_fwd_ms = (_time.perf_counter() - _t0) * 1000.0
_part = _bt.part_from_embeds(_emb, _st.side, _st.uuid(_cells))
print(f"synesthesia: forward {_fwd_ms:.0f} ms -> {tuple(_emb.shape)} {_emb.dtype} "
      f"payload {_se.part_payload_bytes(_part):,} B uuid={_part['uuid']}", flush=True)
assert tuple(_emb.shape) == (1024, 2560) and _emb.dtype == _torch.bfloat16

# the token-estimate fix, measured again against a REAL board-tower part rather than a synthetic one
sys.path.insert(0, str(BUNDLE_DIR / "src/ARC3-Inference"))
from inference.agent import token_estimate as _te
_priced = _te._embeds_part_tokens(_part)
print(f"synesthesia: patched estimator prices the real part at {_priced} tokens (true 1024)", flush=True)
if _priced != 1024:
    raise RuntimeError(f"estimator prices a real student part at {_priced}, not 1024 -- "
                       f"history would be trimmed to nothing every turn")

# the loaded harness module really has the board-tower branch, in THIS process
from inference.agent import vision_context as _vc
assert hasattr(_vc, "_student_tower_part"), "vision_context in sys.modules lacks the board-tower branch"
assert _vc.current_grid_image_enabled(), "MULTIMODAL_CONTEXT is not current_grid -- no board image"
print(f"synesthesia: vision_context has the board-tower branch, current_grid enabled, "
      f"MULTIMODAL_UPSCALE={os.environ.get('MULTIMODAL_UPSCALE')} "
      f"(board channel NATIVE: the PNG path renders at this upscale; the student is loaded only for its self-check)", flush=True)

# ---- the SERVER really got the embedding flags --------------------------------------------------
# Setting ARM_* is not evidence; the launched command line is. Three flags define the channel: without
# --enable-mm-embeds an image_embeds part is rejected, and without image:0 a silent fallback to the
# pixel path would be possible and would look exactly like success.
# A verification must never be the thing that kills an 8.5 h submission: a MISSING watchdog file
# warns, a WRONG flag aborts (a wrong flag means the board channel is not what runs).
_wdp = WORKING_DIR / "vllm-watchdog.json"
_cmd = list(json.loads(_wdp.read_text())["command"]) if _wdp.is_file() else []
if not _cmd:
    print(f"synesthesia: WARNING {_wdp} absent -- cannot verify the serving line", flush=True)
elif "--enable-mm-embeds" not in _cmd:
    raise RuntimeError("server was launched WITHOUT --enable-mm-embeds: image_embeds would 400")
else:
    _lim = json.loads(_cmd[_cmd.index("--limit-mm-per-prompt") + 1])
    if _lim.get("image") != 143:
        raise RuntimeError(f"board channel NATIVE: --limit-mm-per-prompt is {_lim}, not image:143 -- the "
                           f"native tower must be RESIDENT and accept the whole board history per prompt")
    _mpk = json.loads(_cmd[_cmd.index("--mm-processor-kwargs") + 1])
    if _mpk.get("max_pixels") != 1048576:
        raise RuntimeError(f"--mm-processor-kwargs max_pixels={_mpk.get('max_pixels')}, not 1048576 "
                           f"(the board tower produces 1024 tokens/board)")
    print("synesthesia: server flags verified: --enable-mm-embeds, limit-mm image:0, max_pixels 1048576",
          flush=True)
    (ARM_JOB / "SERVER_CMD.json").write_text(json.dumps({"this_run": _cmd}, indent=1))
    print("synesthesia: serving line:", " ".join(_cmd), flush=True)


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None
        # ---- ARC3 (2026-10-18, findings 216d): fit the per-game clock to the time actually left ----
        # The pickled 3,918 s/game x ceil(110/14) = 8 waves ~ 8.7 h assumes a ~15-min load under the 9 h GPU-notebook cap,
        # and a true rerun has no soft end, so a slow load came off the LAST wave. Every game now gets the
        # same clock, sized so all waves end before the cap; a normal load leaves it near 3,918 s.
        _arc3_limit = float(os.environ.get("ARC3_NOTEBOOK_LIMIT_S", "32400"))
        _arc3_margin = float(os.environ.get("ARC3_END_MARGIN_S", "600"))
        _arc3_left = NOTEBOOK_START_EPOCH + _arc3_limit - _arc3_margin - time.time()
        _arc3_waves = -(-len(bm.games) // max(1, int(bm.solver.concurrency or 1)))
        _arc3_fit = _arc3_left / max(1, _arc3_waves)
        _arc3_was = bm.solver.max_runtime_s_per_game
        if _arc3_was is None or _arc3_fit < float(_arc3_was):
            bm.solver.max_runtime_s_per_game = max(600.0, _arc3_fit)
        print(f"ARC3 clock: {len(bm.games)} games / {bm.solver.concurrency} = {_arc3_waves} waves; "
              f"{_arc3_left:.0f} s left -> per-game {_arc3_was} -> {bm.solver.max_runtime_s_per_game}", flush=True)
    else:
        # This build ships no offline env files for the pickled ``__auto__`` specs;
        # on Kaggle, point them at the competition's bundled environment files.
        environment_files_dir = wheelhouse.parent / "environment_files"
        if environment_files_dir.is_dir():
            environments_dir = str(environment_files_dir)
            for game in bm.games:
                if game.arcade_spec.environments_dir == "__auto__":
                    game.arcade_spec = dataclasses.replace(
                        game.arcade_spec, environments_dir=environments_dir
                    )

    # ---- provenance: the EFFECTIVE config, read off the objects the run actually uses ----
    import re as _re
    _student_on = os.environ.get("ARM_B_STUDENT_TOWER", "0") == "1"
    _limit_mm = os.environ.get("ARM_LIMIT_MM") or ""
    _mm_embeds_on = os.environ.get("ARM_ENABLE_MM_EMBEDS", "0") == "1"
    _image_limit_zero = bool(_re.search(r'"image"\s*:\s*0', _limit_mm))
    if _student_on:
        _n_img_tok = int(os.environ.get("ARM_B_EXPECT_NPOS", "0")) or None
        _px = int(round((_n_img_tok ** 0.5) / 2)) if _n_img_tok else None
        _src = "student board tower (image_embeds)"
    else:
        _px = int(os.environ.get("MULTIMODAL_UPSCALE", "16") or 16)
        _n_img_tok = (2 * _px) ** 2
        _src = "PNG pixel path (image_url) through the native tower"
    _tower = ("EXCLUDED - built under no_init_weights on meta device, weights never allocated"
              if (_mm_embeds_on and _image_limit_zero) else "LOADED - native vision tower resident")
    print(f"synesthesia: GEOMETRY px/cell={_px} image_tokens_per_board={_n_img_tok} source={_src}", flush=True)
    print(f"synesthesia: TOWER {_tower}", flush=True)
    print(f"synesthesia: n_games={len(bm.games)} n_passes={bm.n_passes} "
          f"per_game_s={bm.solver.max_runtime_s_per_game} concurrency={bm.solver.concurrency} "
          f"max_actions={bm.solver.max_actions_per_game} soft_end={soft_end}", flush=True)
    (ARM_JOB / "RUN_CONFIG.json").write_text(json.dumps({
        "run": ARM_LABEL,
        "arm": {"title": ARM_NAME, "kernel_id": ARM_KERNEL, "harness_dataset": ARM_HARNESS_DATASET,
                "env": {k: os.environ.get(k) for k in ARM_SWITCHES}},
        "run_as_submission": run_as_submission,
        "competition_rerun": true_submission,
        "games": [g.env_name for g in bm.games], "n_games": len(bm.games),
        "n_passes": bm.n_passes,
        "max_actions_per_game": bm.solver.max_actions_per_game,
        "max_runtime_s_per_game": bm.solver.max_runtime_s_per_game,
        "concurrency": bm.solver.concurrency,
        "analyzer_timeout": bm.solver.analyzer_timeout,
        "save_request_logs": bm.solver.save_request_logs,
        "target_max_runtime_s": float(getattr(target, "max_runtime_s", 0.0) or 0.0),
        "soft_end_utc": soft_end.isoformat() if soft_end else None,
        "schedule_estimate_s": (len(bm.games) / max(1, bm.solver.concurrency))
                               * float(bm.solver.max_runtime_s_per_game or 0.0),
        "perception": {
            "px_per_cell": _px,
            "image_tokens_per_board": _n_img_tok,
            "token_source": _src,
            "vision_tower": _tower,
            "multimodal_context": os.environ.get("MULTIMODAL_CONTEXT"),
            "multimodal_upscale": os.environ.get("MULTIMODAL_UPSCALE"),
            "vllm_max_pixels": os.environ.get("ARM_MAX_PIXELS", "65536 (harness default)"),
            "limit_mm_per_prompt": _limit_mm or '{"video": 0} (harness default)',
            "enable_mm_embeds": _mm_embeds_on,
            "student_ckpt": os.environ.get("BOARD_TOWER_CKPT") if _student_on else None,
            "student_expect_n_pos": os.environ.get("ARM_B_EXPECT_NPOS") if _student_on else None,
            "student_device": os.environ.get("BOARD_TOWER_DEVICE") if _student_on else None,
            "student_threads": os.environ.get("BOARD_TOWER_THREADS") if _student_on else None,
            "student_params": 18221696 if _student_on else None,
        },
        "serving": {
            "max_model_len": bm.solver.kaggle_vllm_max_model_len,
            "max_num_seqs": bm.solver.kaggle_vllm_max_num_seqs,
            "max_num_batched_tokens": bm.solver.kaggle_vllm_max_num_batched_tokens,
            "kv_cache_dtype": bm.solver.kaggle_vllm_kv_cache_dtype,
            "indexer_kv_dtype": bm.solver.kaggle_vllm_indexer_kv_dtype,
            "mamba_ssm_cache_dtype": bm.solver.kaggle_vllm_mamba_ssm_cache_dtype,
            "mtp_tokens": bm.solver.kaggle_vllm_mtp_tokens,
            "prefix_match_unit": bm.solver.kaggle_vllm_prefix_match_unit,
            "prefix_cache_retention_interval": bm.solver.kaggle_vllm_prefix_cache_retention_interval,
            "gpu_memory_utilization": bm.solver.kaggle_vllm_gpu_memory_utilization,
            "quantization": bm.solver.kaggle_vllm_quantization,
            "gdn_decode_kernel": bm.solver.kaggle_vllm_gdn_decode_kernel,
            "tensor_parallel_size": bm.solver.kaggle_vllm_tensor_parallel_size,
            "draft_vocab": bm.solver.kaggle_vllm_draft_vocab,
            "served_model_name": bm.solver.kaggle_served_model_name,
        },
        "host": {"cpu_count": os.cpu_count()},
        "started_utc": datetime.utcnow().isoformat() + "Z",
    }, indent=1))

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        # ---- evidence gate (review finding 1, 2026-10-06): Benchmark.run swallows the
        # soft-deadline cancel and per-game exceptions, so a commit run that never played a single
        # board still COMPLETES and becomes submittable blind (the v7-replicate 0.00 shape). In a
        # NON-submission run, require proof the student path was live: >=1 image_embeds record at
        # 1024 tokens in board_parts.jsonl, written by the patched vision_context. Never fires on a
        # real submission (true_submission), where it would kill a scored run.
        if not true_submission:
            _bp = ARM_JOB / "board_parts.jsonl"
            _n = 0
            _ne = 0
            if _bp.is_file():
                for _ln in _bp.read_text().splitlines():
                    try:
                        _r = json.loads(_ln)
                    except Exception:
                        continue
                    if _r.get("type") == "image_url" and _r.get("n_tokens") == 1024:
                        _n += 1
                    elif _r.get("type") == "image_embeds":
                        _ne += 1
            print(f"synesthesia: evidence gate (board channel NATIVE): {_n} native image_url parts at 1024 "
                  f"tokens, {_ne} image_embeds parts (must be >=1 and 0)", flush=True)
            if _n == 0 or _ne != 0:
                raise RuntimeError(f"board channel NATIVE: evidence gate FAILED: {_n} native parts at 1024 tokens, "
                                   f"{_ne} image_embeds parts -- this run is not the NATIVE arm")
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)